In [17]:
import gymnasium as gym
import numpy as np
from tetris_gymnasium.envs.tetris import Tetris
from tetris_gymnasium.wrappers.observation import RgbObservation, FeatureVectorObservation
from tetris_gymnasium.wrappers.grouped import GroupedActionsObservations
from gymnasium.wrappers import TimeLimit, ResizeObservation, RecordVideo, FrameStackObservation, GrayscaleObservation
from stable_baselines3 import DQN, PPO
import os

In [18]:
RENDER_ENV = False
LOAD_MODEL = False
new_size = (96,136) #(84,84)
batch_size = 32
num_episodes = 468 #43200 #4680 para 1 hora pc Seba
max_episode_steps = 100
num_stacked_frames = 4
intervals = 4
Model = "PPO" # DQN o PPO
version = 3.5

In [19]:
def get_last_modified_file(directory_path):
    if not os.path.isdir(directory_path):
        print(f"Error: Directory '{directory_path}' does not exist.")
        return None
    files = [os.path.join(directory_path, f) for f in os.listdir(directory_path) if os.path.isfile(os.path.join(directory_path, f))]
    if not files:
        return None
    files.sort(key=os.path.getmtime, reverse=True)
    return files[0]

target_directory = f"../Models_Saves/{Model}"  # Replace with your directory path
model_load_path = get_last_modified_file(target_directory)

if model_load_path:
    print(f"The last modified file is: {model_load_path}")
else:
    print("No files found in the directory or directory does not exist.")

The last modified file is: ../Models_Saves/PPO/PPO_V3.5_S468000


In [20]:
try:
    os.mkdir("../Models_Saves")
except Exception as e:
    print(f"Error: {e}")
try:
    os.mkdir("../Models_Saves/PPO")
except Exception as e:
    print(f"Error: {e}")
try:
    os.mkdir("../Models_Saves/DQN")
except Exception as e:
    print(f"Error: {e}")
try:
    os.mkdir("../Video_Tetris_IA")
except Exception as e:
    print(f"Error: {e}")
try:
    os.mkdir("../Video_Tetris_IA/PPO")
except Exception as e:
    print(f"Error: {e}")
try:
    os.mkdir("../Video_Tetris_IA/DQN")
except Exception as e:
    print(f"Error: {e}")

Error: [Errno 17] File exists: '../Models_Saves'
Error: [Errno 17] File exists: '../Models_Saves/PPO'
Error: [Errno 17] File exists: '../Models_Saves/DQN'
Error: [Errno 17] File exists: '../Video_Tetris_IA'
Error: [Errno 17] File exists: '../Video_Tetris_IA/PPO'
Error: [Errno 17] File exists: '../Video_Tetris_IA/DQN'


In [21]:
def calc_max_height(mat):
  mat1 = np.rot90(np.rot90(mat))
  for i in range(len(mat1)):
    count = 0
    for col in mat1[i]:
      if col > 0:
        count+=1
    if count == 0:
      return i-1
  return 19

def calc_holes(mat,height):
  mat1 = np.rot90(np.rot90(np.rot90(mat)))
  holes = 0
  for row in mat1:
    for i in range(height+1):
      if row[i] == 0:
        holes += 1
  return holes

def calc_adj_col(mat, height):
  mat1 = np.rot90(np.rot90(np.rot90(mat)))
  prev_height = -1
  dif_count = 0
  for row in mat1:
    act_height = 0
    for i in range(height+1):
      if row[i] != 0:
        act_height = i+1
    if prev_height >= 0:
      dif_count+=(abs(prev_height-act_height))
    prev_height = act_height
  return dif_count


In [22]:
try:
  env.close()
except:
  print('no hay env para cerrar')

In [ ]:

class CustomRewardWrapper(gym.RewardWrapper):
    #def __init__(self, env, holes_penalty = 0.005, height_penalty = 0.05, increase_height_penalty = 0.4, increase_holes_penalty = 0.2, dif_heigh_penalty = 0.03):
    def __init__(self, env, holes_penalty = 0.000002, height_penalty = 0.00002, increase_height_penalty = 0.4, increase_holes_penalty = 0.2, dif_heigh_penalty = 0.00001):
        super(CustomRewardWrapper, self).__init__(env)
        self.holes_penalty = holes_penalty
        self.height_penalty = height_penalty
        self.increase_height_penalty = increase_height_penalty
        self.increase_holes_penalty = increase_holes_penalty
        self.dif_heigh_penalty = dif_heigh_penalty

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)

        game_variables = self.env.unwrapped.get_state().board[:-4, 4:-4]
        self.previous_max_height = calc_max_height(game_variables)
        self.previous_holes = calc_holes(game_variables, self.previous_max_height)

        return obs, info

    def reward(self, reward):
        #print(f"Reward original: {reward}")
        # Probar mayor penalizacion de agujeros
        custom_reward = reward
        game_variables = self.env.unwrapped.get_state().board[:-4, 4:-4]

        if game_variables.any():
            # Calcula la altura maxima actual
            current_max_height = calc_max_height(game_variables)
            # Calcula ls espacios vacios entre la base y la altura
            current_holes = calc_holes(game_variables, current_max_height)
            #Calcula la diferencia de altura entre las columnas adyacentes y las suma
            current_height_dif = calc_adj_col(game_variables, current_max_height)
            #Penalizacion constante por altura
            custom_reward -= current_max_height*self.height_penalty
            #Penalizacion constante por agujeros
            custom_reward -= current_holes*self.holes_penalty
            #Penalizacion constante por diferencia de altura entre columnas
            custom_reward -= current_height_dif*self.dif_heigh_penalty
            #Penalizacion y recompensa por aumentar altura o disminuir altura respectivamente
            if current_max_height-self.previous_max_height != 0:
                custom_reward -= (current_max_height-self.previous_max_height)*self.increase_height_penalty
            #Penalizacion y recompensa por aumentar agujeros o disminuir agujeros respectivamente
            if current_holes-self.previous_holes != 0:
                custom_reward -= (current_holes-self.previous_holes)*self.increase_holes_penalty
            #Actualiza valores previos para el siguiente paso
            self.previous_holes = current_holes
            self.previous_max_height = current_max_height
        return custom_reward

In [24]:
def make_env(*, game, max_episode_steps=4500, **kwargs):
    env = gym.make(game, **kwargs)
    env = RgbObservation(env)
    #env = FeatureVectorObservation(env)
    env = ResizeObservation(env, new_size)
    env = GrayscaleObservation(env)
    env = FrameStackObservation(env, stack_size=num_stacked_frames)
    env = CustomRewardWrapper(env)
    env.reset(seed=42)
    return env

In [25]:


if __name__ == "__main__":
    env = make_env(game = "tetris_gymnasium/Tetris", render_mode="rgb_array")
    if Model == "DQN":
        model = DQN("CnnPolicy", env, verbose=1, buffer_size=10000) # para Mlp usar FeatureVectorObservation para Cnn usar RgbObservation
    else:
        model = PPO("CnnPolicy", env, verbose=1) # para Mlp usar FeatureVectorObservation para Cnn
    if LOAD_MODEL:
        model.load(model_load_path)
    model.learn(total_timesteps=num_episodes*max_episode_steps, log_interval=4)
    model.save(f"../Models_Saves/{Model}/{Model}_V{version}_S{max_episode_steps*num_episodes}")
    env.close()

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 80.5        |
|    ep_rew_mean          | -26.4       |
| time/                   |             |
|    fps                  | 147         |
|    iterations           | 4           |
|    time_elapsed         | 55          |
|    total_timesteps      | 8192        |
| train/                  |             |
|    approx_kl            | 0.012945199 |
|    clip_fraction        | 0.187       |
|    clip_range           | 0.2         |
|    entropy_loss         | -2.02       |
|    explained_variance   | 0.114       |
|    learning_rate        | 0.0003      |
|    loss                 | 0.556       |
|    n_updates            | 30          |
|    policy_gradient_loss | -0.0184     |
|    value_loss           | 2.5         |
-----------------------------------------
-----------------------

In [26]:
try:
  env = RecordVideo(
    env,
    video_folder=f'../Video_Tetris_IA/{Model}',    # Folder to save videos
    name_prefix=f'{Model}_eval-V{version}-Trained_steps_{num_episodes*max_episode_steps}',               # Prefix for video filenames
    episode_trigger=lambda x: True    # Record every episode
  )
except Exception as e:
  print(f'error implementando grabacion: {e}')

/home/seba/Documentos/AI_Juegos/.venv/lib/python3.10/site-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /home/seba/Documentos/AI_Juegos/Video_Tetris_IA/PPO folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


In [27]:
for episode in range(10):
  state, info = env.reset()
  total_reward = 0
  done = False
  step_count = 0
  while not done:
    step_count+=1
    action, _states = model.predict(state, deterministic=True)
    state, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    total_reward += reward
  print(f"Episode: {episode} Reward: {total_reward} Steps: {step_count}")

Episode: 0 Reward: -22.060809999999996 Steps: 182
Episode: 1 Reward: -16.084013999999975 Steps: 237
Episode: 2 Reward: -18.482076000000003 Steps: 226
Episode: 3 Reward: -22.06361200000001 Steps: 221
Episode: 4 Reward: -18.481559999999995 Steps: 231
Episode: 5 Reward: -22.07035599999999 Steps: 197
Episode: 6 Reward: -8.678770000000007 Steps: 237
Episode: 7 Reward: -16.678280000000015 Steps: 252
Episode: 8 Reward: -20.277428000000018 Steps: 247
Episode: 9 Reward: -22.065816000000005 Steps: 202


In [28]:
# var = len(env.unwrapped.get_state().board)
# count = var
# temp = env.unwrapped.get_state().board[:-4, 4:-4]
# print(temp)
# print(calc_max_height(temp))
# print(calc_holes(temp))
# #print(len(env.unwrapped.get_state().board))

In [29]:
# from stable_baselines3.common.env_checker import check_env
# check_env(env)

In [30]:
env.unwrapped.get_state().board[:-4, 4:-4]

array([[0, 0, 0, 6, 6, 0, 0, 0, 0, 0],
       [0, 0, 3, 3, 6, 6, 0, 0, 0, 0],
       [0, 0, 3, 3, 0, 0, 0, 0, 0, 0],
       [2, 2, 2, 2, 0, 0, 0, 0, 0, 0],
       [0, 7, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 7, 7, 7, 0, 0, 0, 0, 0, 0],
       [0, 4, 0, 0, 0, 0, 0, 0, 0, 0],
       [4, 4, 4, 0, 0, 0, 0, 0, 0, 0],
       [7, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [7, 7, 7, 0, 0, 0, 0, 0, 0, 0],
       [0, 5, 5, 0, 0, 0, 0, 0, 0, 0],
       [5, 5, 0, 0, 0, 0, 0, 0, 0, 0],
       [6, 6, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 6, 6, 0, 0, 0, 0, 0, 0, 0],
       [3, 3, 0, 0, 0, 4, 0, 0, 0, 0],
       [3, 3, 0, 0, 4, 4, 4, 5, 5, 0],
       [2, 2, 2, 2, 0, 4, 5, 5, 0, 0],
       [0, 0, 6, 0, 4, 4, 4, 0, 0, 0],
       [0, 6, 6, 0, 8, 0, 0, 0, 0, 0],
       [0, 6, 8, 8, 8, 0, 0, 0, 0, 0]], dtype=uint8)

In [31]:
np.rot90(np.rot90(np.rot90(env.unwrapped.get_state().board[:-4, 4:-4])))

array([[0, 0, 0, 2, 3, 3, 0, 6, 5, 0, 7, 7, 4, 0, 0, 0, 2, 0, 0, 0],
       [6, 6, 0, 2, 3, 3, 6, 6, 5, 5, 7, 0, 4, 4, 7, 7, 2, 0, 0, 0],
       [8, 6, 6, 2, 0, 0, 6, 0, 0, 5, 7, 0, 4, 0, 7, 0, 2, 3, 3, 0],
       [8, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 2, 3, 3, 6],
       [8, 8, 4, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 6],
       [0, 0, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0],
       [0, 0, 4, 5, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]],
      dtype=uint8)

In [32]:
height_test = calc_max_height(env.unwrapped.get_state().board[:-4, 4:-4])
print(height_test)
print("-"*20)
print(calc_adj_col(env.unwrapped.get_state().board[:-4, 4:-4], height_test))
print("-"*20)
print(height_test)
print(calc_holes(env.unwrapped.get_state().board[:-4, 4:-4], height_test))

19
--------------------
23
--------------------
19
140
